# Quick adapter evaluation

Reuses the base transcripts from the first evaluation, then runs only the trained adapters on
the test split. No judge, no held-out, no controls. Purpose: the first before-and-after number.

In [ ]:
import json, os, shutil, subprocess, sys
from pathlib import Path

def find_repo():
    hits=[p for p in Path("/kaggle/input").rglob("rule_checker.py") if p.exists() and (p.parents[2]/"scripts"/"train.py").exists()]
    hits.sort(key=lambda p: len(p.parts))
    if not hits: raise SystemExit("code not mounted")
    return hits[0].parents[2]

WORK=Path("/kaggle/working/htd")
if WORK.exists(): shutil.rmtree(WORK)
shutil.copytree(find_repo(), WORK)
os.chdir(WORK); sys.path.insert(0,str(WORK/"src"))
for p in Path("/kaggle/input").rglob("hf_token.txt"):
    os.environ.setdefault("HF_TOKEN", p.read_text().strip())
    os.environ.setdefault("HUGGINGFACE_API_KEY", p.read_text().strip())
assert os.environ.get("HF_TOKEN")
FIRE="CohereLabs/tiny-aya-fire"
print("code:", WORK)

def run(cmd, **kw):
    print("$", " ".join(str(c) for c in cmd), flush=True)
    return subprocess.run(cmd, check=True, **kw)


In [ ]:
base = next(Path("/kaggle/input").rglob("fire_test_plain.jsonl")).parent
Path("outputs/eval").mkdir(parents=True, exist_ok=True)
for p in base.glob("fire_*.jsonl"):
    shutil.copy(p, f"outputs/eval/{p.name}")
print("base transcripts:", sorted(p.name for p in Path('outputs/eval').glob('fire_*.jsonl')))

In [ ]:
run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"])

In [ ]:
def adapter_path(name):
    for p in Path("/kaggle/input").rglob("adapter_config.json"):
        if p.parent.name == name:
            return str(p.parent)
    return None

FOUND = {"at": adapter_path("aligntune_dpo"), "sft": adapter_path("sft_lr1e4"),
         "dpo5": adapter_path("dpo_b01_lr5e6")}
FOUND = {k:v for k,v in FOUND.items() if v}
print(json.dumps(FOUND, indent=2))
assert FOUND, "no adapters"

In [ ]:
run([sys.executable, "scripts/build_pilot.py", "--split", "test", "--n-per-script", "20", "--out", "data/generated/test_dialogues.jsonl"])

In [ ]:
files=["outputs/eval/fire_test_plain.jsonl"]+[f"outputs/eval/{k}_test.jsonl" for k in FOUND]
subprocess.run(["python","scripts/score_run.py",*files,"--out","outputs/eval/summary_test.json"],check=True)
for k in FOUND:
    subprocess.run(["python","scripts/compare.py","outputs/eval/fire_test_plain.jsonl",
                    f"outputs/eval/{k}_test.jsonl","--out",f"outputs/eval/cmp_{k}_test.json"],check=True)

In [ ]:
print(Path("outputs/eval/summary_test.json").read_text())